# PINN Forward Notebook



# Forward PINN for the Malaria Transmission PDE System

This notebook implements a **forward Physics-Informed Neural Network (PINN)** for the malaria transmission model defined over the variables **time** (t), **age** (a), and **exposure** (\xi).
The objective is to approximate the solution of the full PDE system generated by the numerical solver while preserving the original computational workflow of the notebook.

## Scientific scope

The notebook:

* loads the reference numerical solution from `malaria_simulation_results.npz`;
* trains a forward PINN on the PDE posed in ((t,a,\xi));
* combines data fitting, PDE residual minimization, initial condition enforcement, age-inflow constraints, and Neumann boundary conditions in (\xi);
* evaluates the trained model with detailed residual and error diagnostics;
* produces article-ready visualizations for the five compartments (S, D, T, A, P).


## 1. Data upload

Upload the reference NPZ file generated by the numerical solver before running the training pipeline.

In [ ]:
from google.colab import files
uploaded = files.upload()   # boîte de dialogue pour choisir le .npz

## 2. Computational environment and imports

This section initializes the runtime environment, imports the required libraries, and fixes the random seed for reproducibility.

In [ ]:
# Colab: torch is already available. Otherwise:
# !pip -q install torch --extra-index-url https://download.pytorch.org/whl/cu121
import os, math, random, json, time, bisect
import numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt

# Reproducibility
SEED = 2025
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# PyTorch determinism (may raise errors if a non-deterministic op is used)
torch.use_deterministic_algorithms(True)

## 3. Loading reference data and normalization constants

The reference solution generated by the numerical solver is loaded from the NPZ archive using memory mapping to remain compatible with low-RAM environments. Component-wise output statistics are estimated for standardized training.

In [ ]:
NPZ_PATH = "malaria_simulation_results.npz"  #  Make sure to upload the file to Colab
assert os.path.isfile(NPZ_PATH), "Upload 'malaria_simulation_results.npz' in the working directory."

Z = np.load(NPZ_PATH, allow_pickle=True, mmap_mode='r')
U_snaps = Z["U_snaps"]    # shape: (n_snapshots, Na_pts, Nx, 5)
times   = Z["times"]      # (n_snapshots,)
age_grid= Z["age_grid"]   # (Na_pts,)
xi_grid = Z["xi_grid"]    # (Nx,)
mu_true = Z["mu_true"]  # [d_exp, v_exp]
print("U_snaps:", U_snaps.shape, "| times:", times.shape, "| age:", age_grid.shape, "| xi:", xi_grid.shape)
print("mu_true: d_exp=%.5f, v_exp=%.5f"%(mu_true[0], mu_true[1]))

t_min, t_max = float(times.min()), float(times.max())
a_min, a_max = float(age_grid.min()), float(age_grid.max())
x_min, x_max = float(xi_grid.min()), float(xi_grid.max())

# Output statistics (component-wise standardization) on a subsample (RAM-friendly)
it = np.linspace(0, len(times)-1, min(20, len(times)), dtype=int)
ia = np.linspace(0, len(age_grid)-1, min(40, len(age_grid)), dtype=int)
ix = np.linspace(0, len(xi_grid)-1, min(40, len(xi_grid)), dtype=int)
buf = np.concatenate([U_snaps[i][np.ix_(ia, ix)].reshape(-1,5) for i in it], axis=0)
y_mean = torch.tensor(buf.mean(0), dtype=torch.float32, device=device)
y_std  = torch.tensor(buf.std(0)+1e-6, dtype=torch.float32, device=device)

def scale_in(t,a,x):
    t_=(t-t_min)/(t_max-t_min+1e-12); a_=(a-a_min)/(a_max-a_min+1e-12); x_=(x-x_min)/(x_max-x_min+1e-12)
    return t_,a_,x_

def unscale_in(t_,a_,x_):
    return (t_*(t_max-t_min)+t_min, a_*(a_max-a_min)+a_min, x_*(x_max-x_min)+x_min)


## 4. Fixed parameters, reaction operator, and inflow reconstruction

This section defines the fixed epidemiological parameters and the reaction matrix, denoted in the article by (A_m). It also reconstructs the inflow function (b(t)) from the numerical snapshots at (a=0).

In [ ]:
params = {"Lambda":0.35,"phi":0.4,"f_T":0.5,"d_D":5.0,"d_T":5.0,"d_A":110.0,"d_P":25.0}
d_exp, v_exp = float(mu_true[0]), float(mu_true[1])

def A_matrix(p, device=device):
    La,ph,fT = [torch.tensor(p[k],device=device) for k in ["Lambda","phi","f_T"]]
    dD,dT,dA,dP = [torch.tensor(p[k],device=device) for k in ["d_D","d_T","d_A","d_P"]]
    A = torch.zeros(5,5,device=device)
    A[0,0]=-La; A[0,3]=1/dA; A[0,4]=1/dP
    A[1,0]=ph*(1-fT)*La; A[1,1]=-1/dD; A[1,3]=ph*(1-fT)*La
    A[2,0]=ph*fT*La; A[2,2]=-1/dT; A[2,3]=ph*fT*La
    A[3,0]=(1-ph)*La; A[3,1]=1/dD; A[3,3]=-(ph*La+1/dA)
    A[4,2]=1/dT; A[4,4]=-1/dP
    return A

A_fixed = A_matrix(params)

# b(t) reconstructed from the NPZ file: spatial mean of S(t, a=0, xi)
def b_from_npz(tval: float):
    # Linear interpolation in time
    if tval <= times[0]: return float(U_snaps[0,0,:,0].mean())
    if tval >= times[-1]: return float(U_snaps[-1,0,:,0].mean())
    i = max(0, min(len(times)-2, int(np.searchsorted(times, tval)-1)))
    w = (tval-times[i])/(times[i+1]-times[i]+1e-12)
    return float((1-w)*U_snaps[i,0,:,0].mean() + w*U_snaps[i+1,0,:,0].mean())

phi0_np = U_snaps[0]  # snapshot at t=0

## 5. Forward PINN architecture

The neural network combines Fourier feature encoding with a multilayer perceptron to improve the approximation of structured solutions in ((t,a,\xi)).

In [ ]:
import torch.nn.functional as F

class FourierFeatures(nn.Module):
    """
    Simple, stable, and low-cost positional encoding.
    For each scalar z∈[0,1]: concatenates [z, sin(2π f z), cos(2π f z)] for f=1..K.
    """
    def __init__(self, K=4):
        super().__init__()
        self.K = K
    def forward(self, z):  # (N,1)
        # z in [0,1]
        feats = [z]
        for f in range(1, self.K+1):
            w = 2*math.pi*f
            feats += [torch.sin(w*z), torch.cos(w*z)]
        return torch.cat(feats, dim=1)  # (N, 1+2K)

class EncodedMLP(nn.Module):
    def __init__(self, width=128, depth=5, act="tanh", K=4):
        super().__init__()
        self.enc = FourierFeatures(K=K)
        in_dim = 3*(1+2*K)  # encoded t_, a_, x_
        actl = {"tanh": nn.Tanh(), "relu": nn.ReLU(), "silu": nn.SiLU()}[act]
        layers = [nn.Linear(in_dim, width), actl]
        for _ in range(depth-1):
            layers += [nn.Linear(width, width), actl]
        layers += [nn.Linear(width, 5)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):  # x: (N,3) = [t_,a_,x_]
        t_, a_, x_ = x[:, :1], x[:, 1:2], x[:, 2:3]
        te = self.enc(t_); ae = self.enc(a_); xe = self.enc(x_)
        z = torch.cat([te, ae, xe], dim=1)
        return self.net(z)

# Replaces the former MLP:
model = EncodedMLP(width=128, depth=5, act="tanh", K=4).to(device)

# Learned uncertainty is removed here (it often downweights BC terms)
#     and explicit fixed weights will be used in TRAINING
# (log_sigma2 removed / EMA is still available if needed)
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay=decay
        self.shadow={k:v.detach().clone() for k,v in model.state_dict().items()}
    def update(self, model):
        for k,v in model.state_dict().items():
            self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1-self.decay)
    def apply_to(self, model):
        model.load_state_dict(self.shadow, strict=False)

ema = EMA(model)
amp = torch.cuda.amp.autocast if device.type=="cuda" else (lambda: torch.enable_grad())

## 6. PDE residual and sampling strategy

The residual is computed for the original PDE formulation in ((t,a,\xi)). Sampling is deliberately biased toward boundary and near-boundary regions to better enforce the initial, inflow, and Neumann constraints.

In [ ]:
def residual(t_, a_, x_):
    """
    Split-grads derivatives (Ut, Ua, Ux) + Uxx.
    We keep your formulation (non-constraining) to remain RAM-stable and efficient.
    """
    t_rng = (t_max - t_min) + 1e-12
    a_rng = (a_max - a_min) + 1e-12
    x_rng = (x_max - x_min) + 1e-12

    # ---- Pass 1: dU/dt_
    t_t = t_.clone().requires_grad_(True)
    a_t = a_.detach()
    x_t = x_.detach()
    U_std_t = model(torch.cat([t_t, a_t, x_t], 1))
    Ut_list=[]
    for k in range(5):
        gk = torch.autograd.grad(U_std_t[:, k].sum(), t_t, create_graph=True)[0]
        Ut_list.append(gk)
    Ut_norm = torch.cat(Ut_list, dim=1)
    Ut = (Ut_norm * y_std) / t_rng

    # ---- Pass 2: dU/da_
    t_a = t_.detach()
    a_a = a_.clone().requires_grad_(True)
    x_a = x_.detach()
    U_std_a = model(torch.cat([t_a, a_a, x_a], 1))
    Ua_list=[]
    for k in range(5):
        gk = torch.autograd.grad(U_std_a[:, k].sum(), a_a, create_graph=True)[0]
        Ua_list.append(gk)
    Ua_norm = torch.cat(Ua_list, dim=1)
    Ua = (Ua_norm * y_std) / a_rng

    # ---- Pass 3: dU/dx_ + d²U/dx_²
    t_x = t_.detach()
    a_x = a_.detach()
    x_x = x_.clone().requires_grad_(True)
    U_std_x = model(torch.cat([t_x, a_x, x_x], 1))
    Ux_list=[]
    for k in range(5):
        gk = torch.autograd.grad(U_std_x[:, k].sum(), x_x, create_graph=True)[0]
        Ux_list.append(gk)
    Ux_norm = torch.cat(Ux_list, dim=1)
    Ux  = (Ux_norm * y_std) / x_rng

    Uxx_list=[]
    for k in range(5):
        g2k = torch.autograd.grad(Ux_norm[:, k].sum(), x_x, create_graph=True)[0]
        Uxx_list.append(g2k)
    Uxx_norm = torch.cat(Uxx_list, dim=1)
    Uxx = (Uxx_norm * y_std) / (x_rng**2)

    # ---- Physical U
    U = U_std_x * y_std + y_mean
    r = Ut + Ua - d_exp * Uxx + v_exp * Ux - (U @ A_fixed.T)
    return r, U_std_x

def pick_grid(n):
    it = np.random.randint(0,len(times),size=n)
    ia = np.random.randint(0,len(age_grid),size=n)
    ix = np.random.randint(0,len(xi_grid),size=n)
    t = torch.tensor((times[it]-t_min)/(t_max-t_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    a = torch.tensor((age_grid[ia]-a_min)/(a_max-a_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    x = torch.tensor((xi_grid[ix]-x_min)/(x_max-x_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    U = torch.tensor(U_snaps[it,ia,ix,:],dtype=torch.float32,device=device)
    U_std = (U - y_mean)/y_std
    return t,a,x,U_std

# --- IC (t=0) ---
def batch_IC(n):
    ia = np.random.randint(0,len(age_grid),size=n)
    ix = np.random.randint(0,len(xi_grid),size=n)
    t0 = torch.zeros(n,1,device=device)
    a = torch.tensor((age_grid[ia]-a_min)/(a_max-a_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    x = torch.tensor((xi_grid[ix]-x_min)/(x_max-x_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    U = torch.tensor(phi0_np[ia,ix,:],dtype=torch.float32,device=device)
    return t0,a,x,(U - y_mean)/y_std

# --- Inflow (a=0) ---
def batch_A0(n):
    t = torch.rand(n,1,device=device)
    a0 = torch.zeros_like(t)
    x = torch.rand(n,1,device=device)
    t_phys = (t*(t_max-t_min)+t_min).flatten().tolist()
    b_vals = torch.tensor([b_from_npz(tt) for tt in t_phys],dtype=torch.float32,device=device).reshape(-1,1)
    U = torch.cat([b_vals, torch.zeros_like(b_vals).repeat(1,4)],1)
    return t,a0,x,(U - y_mean)/y_std

# --- Interior collocation, BIASED toward the boundaries (t≈0, a≈0, x≈{0,1}) ---
def batch_phys(n, edge_bias=0.6):
    """
    edge_bias in [0,1]: proportion of points sampled with a Beta(0.5,0.5) law (concentrated near the boundaries)
    for t and a; for x, half are placed on the boundaries and half are sampled uniformly.
    """
    nb = int(n*edge_bias)
    nu = n - nb

    # boundary-focused via Beta(0.5,0.5)
    def beta_edge(m):
        z = torch.distributions.Beta(0.5,0.5).sample((m,1)).to(device)
        return z.clamp(0,1)

    t_b = beta_edge(nb); a_b = beta_edge(nb)
    # x_b: half at 0, half at 1
    k = nb//2
    x_b = torch.cat([torch.zeros(k,1,device=device), torch.ones(nb-k,1,device=device)], 0)

    # uniform
    t_u = torch.rand(nu,1,device=device)
    a_u = torch.rand(nu,1,device=device)
    x_u = torch.rand(nu,1,device=device)

    t = torch.cat([t_b,t_u],0); a = torch.cat([a_b,a_u],0); x = torch.cat([x_b,x_u],0)
    # shuffle
    idx = torch.randperm(n, device=device)
    return t[idx], a[idx], x[idx]

def batch_Neumann(n):
    t = torch.rand(n,1,device=device)
    a = torch.rand(n,1,device=device)
    x_b = torch.cat([torch.zeros(n//2,1,device=device),
                     torch.ones(n - n//2,1,device=device)], 0).requires_grad_(True)
    inp = torch.cat([t, a, x_b], 1)
    U_std = model(inp)
    dUdx_list = []
    for k in range(5):
        gk = torch.autograd.grad(U_std[:,k].sum(), x_b, create_graph=True)[0]
        dUdx_list.append(gk)
    dUdx = torch.cat(dUdx_list, dim=1)
    return dUdx

## 7. Training

In [ ]:
BPhys=4096; BData=2048; BIC=2048; BA0=4096; BNeu=2048
EPOCHS=5000
val_every=1000; patience=8; best=float("inf"); bad=0

# Fixed weights
W_DATA = 1.0
W_PDE  = 2.0     #
W_IC   = 6.0
W_A0   = 10.0    #
W_NEU  = 3.0
W_INFLAT = 0.5   # ξ-flatness penalty at a=0 (S component)
L2_REG = 1e-7    # light regularization

opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=4000, eta_min=2e-4)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type=="cuda"))
history=[]

def step_losses(edge_bias=0.6):
    # --- Data (FD)
    t,a,x,Ud = pick_grid(BData)
    r,Up = residual(t,a,x)                  # Up = standardized prediction
    Ld = ((Up-Ud)**2).mean()

    # --- Physics (residual) — boundary-biased collocation
    t,a,x = batch_phys(BPhys, edge_bias=edge_bias)
    r,_ = residual(t,a,x)
    Lp = (r**2).mean()

    # --- IC (t=0)
    t,a,x,U0 = batch_IC(BIC)
    _,Up_ic = residual(t,a,x)
    Lic = ((Up_ic-U0)**2).mean()

    # --- Inflow (a=0)
    t,a,x,Ua = batch_A0(BA0)
    _,Up_a0 = residual(t,a,x)
    La0 = ((Up_a0-Ua)**2).mean()

    # --- Inflow "flatness" in ξ for S (component 0), in physical units
    #     We want S(t, a=0, ξ) ≈ b(t), constant in ξ.
    S_phys = Up_a0[:,0]*y_std[0] + y_mean[0]          # (BA0,)
    S_mean = S_phys.mean()
    L_inflat = torch.mean((S_phys - S_mean)**2)       # variance in ξ

    # --- Neumann (x=0/1)
    dUdx = batch_Neumann(BNeu)
    Lneu = (dUdx**2).mean()

    # --- small L2 regularization of standardized outputs (stability)
    with torch.no_grad():
        pass
    z = torch.rand(2048,3,device=device)
    y = model(z)
    L_reg = (y**2).mean()

    return Ld, Lp, Lic, La0, L_inflat, Lneu, L_reg

print("Training...")
t0 = time.time()
for ep in range(1, EPOCHS+1):
    # Boundary curriculum: keep strong boundary bias slightly longer (helps IC/IN)
    if ep < 2500:
        edge_bias = 0.8
        Wic, Wa0, Wneu = W_IC, W_A0, W_NEU
    elif ep < 4000:
        edge_bias = 0.7
        Wic, Wa0, Wneu = max(4.0, W_IC*0.8), max(6.0, W_A0*0.8), max(2.0, W_NEU*0.8)
    else:
        edge_bias = 0.6
        Wic, Wa0, Wneu = 3.0, 5.0, 2.0

    opt.zero_grad(set_to_none=True)
    with amp():
        Ld,Lp,Lic,La0,Linflat,Ln,Lreg = step_losses(edge_bias=edge_bias)
        tot = (W_DATA*Ld + W_PDE*Lp + Wic*Lic + Wa0*La0 + W_INFLAT*Linflat + Wneu*Ln + L2_REG*Lreg)
    scaler.scale(tot).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update()
    sched.step(); ema.update(model)

    if ep%500==0 or ep==1:
        val = float((Ld+Lp+Lic+La0+Ln+Linflat).detach().cpu())
        history.append([ep, float(Ld), float(Lp), float(Lic), float(La0), float(Ln), float(Linflat), val])
        dt = time.time()-t0; t0=time.time()
        print(f"Ep {ep:4d} | data {Ld:.2e} phys {Lp:.2e} ic {Lic:.2e} a0 {La0:.2e} inflat {Linflat:.2e} neu {Ln:.2e} | val {val:.2e} | {dt:.1f}s")

    if ep%val_every==0:
        val = float((Ld+Lp+Lic+La0+Ln+Linflat).detach().cpu())
        if val < best - 1e-6:
            best = val; bad = 0
            torch.save({
                "model": model.state_dict(),
                "y_mean": y_mean.detach().cpu().numpy().tolist(),
                "y_std":  y_std.detach().cpu().numpy().tolist()
            }, "forward_best.pth")
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping."); break

# Save the training history for the automatic report
with open("forward_history.json","w") as f:
    json.dump([{"epoch":int(e),"L_data":float(d),"L_phys":float(p),"L_ic":float(ic),
                "L_a0":float(a0),"L_inflat":float(lf),"L_neu":float(neu),"val":float(v)}
               for e,d,p,ic,a0,lf,neu,v in history], f)
print("Saved: forward_best.pth, forward_history.json")

## 8. Evaluation

In [ ]:
# --- sampling utilities (normalized inputs) ---
def _sample_interior(n):
    t = torch.rand(n,1,device=device)
    a = torch.rand(n,1,device=device)
    x = torch.rand(n,1,device=device)
    return t,a,x

def _sample_boundary_neumann(n):
    t = torch.rand(n,1,device=device)
    a = torch.rand(n,1,device=device)
    x_b = torch.cat([torch.zeros(n//2,1,device=device),
                     torch.ones(n - n//2,1,device=device)], 0)
    return t,a,x_b

# --- physical derivatives Ut, Ua, Ux via split-grads; Uxx via centered finite differences ---
def _pde_terms_split(t_, a_, x_, eps_x=1e-3):
    """
    Computes (Ut, Ua, Ux, Uxx, U) in physical (de-standardized) units.
    - Ut, Ua, Ux: autograd, computed on separate graphs (split-grads)
      with retain_graph=True for all components except the last one.
    - Uxx       : centered finite difference in x_ (avoids second backpropagation).
    """
    t_rng = (t_max - t_min) + 1e-12
    a_rng = (a_max - a_min) + 1e-12
    x_rng = (x_max - x_min) + 1e-12

    # ---- Pass 1: Ut (dU/dt_)
    t_t = t_.clone().requires_grad_(True); a_t = a_.detach(); x_t = x_.detach()
    Ustd_t = model(torch.cat([t_t,a_t,x_t],1))
    Ut_list=[]
    for k in range(5):
        gk = torch.autograd.grad(
            Ustd_t[:,k].sum(), t_t,
            create_graph=False,
            retain_graph=(k < 4)
        )[0]
        Ut_list.append(gk)
    Ut_norm = torch.cat(Ut_list,1)             # (N,5)
    Ut = (Ut_norm * y_std) / t_rng             # non-standardized

    # ---- Pass 2: Ua (dU/da_)
    t_a = t_.detach(); a_a = a_.clone().requires_grad_(True); x_a = x_.detach()
    Ustd_a = model(torch.cat([t_a,a_a,x_a],1))
    Ua_list=[]
    for k in range(5):
        gk = torch.autograd.grad(
            Ustd_a[:,k].sum(), a_a,
            create_graph=False,
            retain_graph=(k < 4)
        )[0]
        Ua_list.append(gk)
    Ua_norm = torch.cat(Ua_list,1)
    Ua = (Ua_norm * y_std) / a_rng

    # ---- Pass 3: Ux (dU/dx_)
    t_x = t_.detach(); a_x = a_.detach(); x_x = x_.clone().requires_grad_(True)
    Ustd_x = model(torch.cat([t_x,a_x,x_x],1))
    Ux_list=[]
    for k in range(5):
        gk = torch.autograd.grad(
            Ustd_x[:,k].sum(), x_x,
            create_graph=False,
            retain_graph=(k < 4)
        )[0]
        Ux_list.append(gk)
    Ux_norm = torch.cat(Ux_list,1)
    Ux = (Ux_norm * y_std) / x_rng

    # ---- U (non-standardized) — reuse pass 3, then detach
    U = (Ustd_x * y_std + y_mean).detach()

    # ---- Uxx (d²U/dx²) by centered finite difference on x_ (no autograd)
    x_plus  = torch.clamp(x_.detach() + eps_x, 0.0, 1.0)
    x_minus = torch.clamp(x_.detach() - eps_x, 0.0, 1.0)
    with torch.no_grad():
        Ustd_p = model(torch.cat([t_.detach(), a_.detach(), x_plus], 1))
        Ustd_m = model(torch.cat([t_.detach(), a_.detach(), x_minus],1))
    Uxx_norm = (Ustd_p - 2.0*Ustd_x.detach() + Ustd_m) / (eps_x**2)
    Uxx = (Uxx_norm * y_std) / (x_rng**2)

    return Ut, Ua, Ux, Uxx, U

# --- DATA error (non-standardized) ---
@torch.no_grad()
def rel_err_denorm(n=5000):
    it=np.random.randint(0,len(times),size=n)
    ia=np.random.randint(0,len(age_grid),size=n)
    ix=np.random.randint(0,len(xi_grid),size=n)
    t=torch.tensor((times[it]-t_min)/(t_max-t_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    a=torch.tensor((age_grid[ia]-a_min)/(a_max-a_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    x=torch.tensor((xi_grid[ix]-x_min)/(x_max-x_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    target=torch.tensor(U_snaps[it,ia,ix,:],dtype=torch.float32,device=device)
    pred_std=model(torch.cat([t,a,x],1))
    pred = pred_std * y_std + y_mean
    num=torch.sqrt(((pred-target)**2).sum(1)).mean()
    den=torch.sqrt((target**2).sum(1)).mean()+1e-9
    return float((num/den*100).item())

# --- PDE error (absolute L2 and relative %) ---
def pde_error(n=2000, eps_x=1e-3):
    t,a,x = _sample_interior(n)
    with torch.enable_grad():
        Ut,Ua,Ux,Uxx,U = _pde_terms_split(t,a,x,eps_x=eps_x)
        AU = U @ A_fixed.T
        r = Ut + Ua - d_exp*Uxx + v_exp*Ux - AU                    # (N,5)
        abs_err = torch.sqrt((r**2).sum(1)).mean().item()
        ref = (Ut.abs() + Ua.abs() + (d_exp*Uxx).abs() + (v_exp*Ux).abs() + AU.abs()).sum(1).mean().item() + 1e-9
        rel_pct = (torch.sqrt((r**2).sum(1)).mean() / ref) * 100.0
    return float(abs_err), float(rel_pct.item())

# --- Neumann error (absolute + relative vs interior |Ux|) ---
def neumann_error(n=2000):
    x_rng = (x_max - x_min) + 1e-12
    with torch.enable_grad():
        t,a,x_b = _sample_boundary_neumann(n)
        x_b = x_b.clone().requires_grad_(True)
        Ustd = model(torch.cat([t,a,x_b],1))
        dUdx_list=[]
        for k in range(5):
            gk = torch.autograd.grad(
                Ustd[:,k].sum(), x_b,
                create_graph=False,
                retain_graph=(k < 4)
            )[0]
            dUdx_list.append(gk)
        dUdx_norm = torch.cat(dUdx_list,1)
        dUdx_phys = (dUdx_norm * y_std) / x_rng
        abs_neu = torch.sqrt((dUdx_phys**2).sum(1)).mean().item()

        # reference: mean norm of interior |Ux|
        ti,ai,xi = _sample_interior(n)
        Ut,Ua,Ux,_,_ = _pde_terms_split(ti,ai,xi)
        ref = torch.sqrt((Ux**2).sum(1)).mean().item() + 1e-9
        rel_pct = (torch.sqrt((dUdx_phys**2).sum(1)).mean() / ref) * 100.0
    return float(abs_neu), float(rel_pct.item())

# --- IC error (t=0) and inflow error (a=0), non-standardized ---
@torch.no_grad()
def ic_error(n=4000):
    ia=np.random.randint(0,len(age_grid),size=n)
    ix=np.random.randint(0,len(xi_grid),size=n)
    t0=torch.zeros(n,1,device=device)
    a=torch.tensor((age_grid[ia]-a_min)/(a_max-a_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    x=torch.tensor((xi_grid[ix]-x_min)/(x_max-x_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    target=torch.tensor(phi0_np[ia,ix,:],dtype=torch.float32,device=device)
    pred = model(torch.cat([t0,a,x],1))*y_std + y_mean
    num = torch.sqrt(((pred-target)**2).sum(1)).mean()
    den = torch.sqrt((target**2).sum(1)).mean()+1e-9
    return float((num/den*100).item())

@torch.no_grad()
def inflow_error(n=4000):
    t=torch.rand(n,1,device=device); a0=torch.zeros_like(t); x=torch.rand(n,1,device=device)
    t_phys=(t*(t_max-t_min)+t_min).flatten().tolist()
    b_vals=torch.tensor([b_from_npz(tt) for tt in t_phys],dtype=torch.float32,device=device).reshape(-1,1)
    target=torch.cat([b_vals, torch.zeros_like(b_vals).repeat(1,4)],1)
    pred = model(torch.cat([t,a0,x],1))*y_std + y_mean
    num = torch.sqrt(((pred-target)**2).sum(1)).mean()
    den = torch.sqrt((target**2).sum(1)).mean()+1e-9
    return float((num/den*100).item())

# ---- Run the metrics  ----
err_data = rel_err_denorm(4000)
pde_abs, pde_rel = pde_error(n=1500, eps_x=1e-3)
neu_abs, neu_rel = neumann_error(n=1500)
ic_rel = ic_error(4000)
in_rel = inflow_error(4000)

print(f"[DATA]   Relative error (non-std): {err_data:.2f}%")
print(f"[PDE]    Absolute residual (L2): {pde_abs:.3e} | Relative residual: {pde_rel:.2f}%")
print(f"[NEU]    Absolute Neumann error: {neu_abs:.3e} | Relative Neumann error (vs interior |Ux|): {neu_rel:.2f}%")
print(f"[IC]     IC error (t=0): {ic_rel:.2f}%")
print(f"[INFLOW] Inflow error (a=0): {in_rel:.2f}%")

## 9. Visualisations and comparaison with numerical solver results


In [ ]:
import math
import matplotlib.pyplot as plt
from matplotlib import cm

# 1) Save the forward PINN under the requested name
torch.save({
    "model": model.state_dict(),
    "y_mean": y_mean.detach().cpu().numpy().tolist(),
    "y_std":  y_std.detach().cpu().numpy().tolist()
}, "PINN_forward.pth")
print("Saved model as: PINN_forward.pth")

comp_names = ['S','D','T','A','P']  # order of the 5 channels
K = len(comp_names)

# ---------- utilities ----------
@torch.no_grad()
def predict_U_phys_on_grid(t_phys, A=None, X=None, chunk=20000):
    """
    Returns U_pred(a,xi,5) in physical units at a given time t_phys.
    A and X are numpy vectors (default: full age_grid/xi_grid grids)
    """
    if A is None: A = age_grid
    if X is None: X = xi_grid
    a = torch.tensor((A - a_min)/((a_max-a_min)+1e-12), dtype=torch.float32, device=device).reshape(-1,1)
    x = torch.tensor((X - x_min)/((x_max-x_min)+1e-12), dtype=torch.float32, device=device).reshape(-1,1)
    t_ = torch.full((a.shape[0]*x.shape[0],1), (t_phys - t_min)/((t_max - t_min)+1e-12), device=device)
    aa, xx = torch.meshgrid(a.flatten(), x.flatten(), indexing='ij')
    inp = torch.cat([t_, aa.reshape(-1,1), xx.reshape(-1,1)], 1)

    out = []
    for i in range(0, inp.shape[0], chunk):
        pred_std = model(inp[i:i+chunk])
        pred = pred_std * y_std + y_mean
        out.append(pred.detach().cpu())
    U = torch.cat(out,0).numpy().reshape(len(A), len(X), K)
    return U  # (Na, Nx, 5)

def nearest_time_index(t_phys):
    return int(np.abs(times - t_phys).argmin())

# ---------- 2) Curves U_k(a, xi=xi_mid) for several times ----------
def plot_lines_vs_age_times(t_list=None, xi_sel=None, fname="viz_lines_pred.png"):
    if t_list is None:
        t_list = [float(times.min()), float(0.25*(times.min()+times.max())),
                  float(0.5*(times.min()+times.max())),
                  float(0.75*(times.min()+times.max())),
                  float(times.max())]
    if xi_sel is None:
        xi_sel = float(0.5*(x_min+x_max))  # xi at the center

    # grids
    A = age_grid.copy()
    fig, axes = plt.subplots(1, K, figsize=(20,3), sharex=True, sharey=False)
    for j in range(K):
        ax = axes[j]
        for t_phys in t_list:
            U = predict_U_phys_on_grid(t_phys, A=A, X=np.array([xi_sel]))
            y = U[:,0,j]  # (Na,)
            ax.plot(A, y, label=f"t={ (t_phys-t_min)/(t_max-t_min+1e-12):.2f}")
        ax.set_title(f"{comp_names[j]}(a, ξ={xi_sel:.2f}) — PINN")
        ax.set_xlabel("a");
        if j==0: ax.set_ylabel("value")
        ax.grid(True, alpha=0.3)
    axes[0].legend(ncol=1, fontsize=8, frameon=False)
    plt.tight_layout()
    plt.savefig(fname, dpi=160, bbox_inches='tight')
    plt.show()
    print("Saved:", fname)

# ---------- 3) Comparison curves (True vs Pred) ----------
def plot_lines_vs_age_times_compare(t_list=None, xi_sel=None, fname="viz_lines_compare.png"):
    if t_list is None:
        t_list = [float(0.25*(t_min+t_max)), float(0.5*(t_min+t_max)), float(0.75*(t_min+t_max))]
    if xi_sel is None:
        xi_sel = float(0.5*(x_min+x_max))
    jxi = int(np.abs(xi_grid - xi_sel).argmin())

    A = age_grid.copy()
    fig, axes = plt.subplots(1, K, figsize=(20,3), sharex=True, sharey=False)
    for j in range(K):
        ax = axes[j]
        for t_phys in t_list:
            # pred
            Up = predict_U_phys_on_grid(t_phys, A=A, X=np.array([xi_sel]))[:,0,j]
            # true
            it = nearest_time_index(t_phys)
            Utrue = U_snaps[it,:,jxi,j]
            ax.plot(A, Utrue, '--', lw=1.2, label=f"True t={ (t_phys-t_min)/(t_max-t_min+1e-12):.2f}")
            ax.plot(A, Up,    '-',  lw=1.6, label=f"Pred t={ (t_phys-t_min)/(t_max-t_min+1e-12):.2f}")
        ax.set_title(f"{comp_names[j]}(a, ξ={xi_sel:.2f}) — True vs Pred")
        ax.set_xlabel("a")
        if j==0: ax.set_ylabel("value")
        ax.grid(True, alpha=0.3)
    axes[0].legend(ncol=2, fontsize=8, frameon=False)
    plt.tight_layout()
    plt.savefig(fname, dpi=160, bbox_inches='tight')
    plt.show()
    print("Saved:", fname)

# ---------- 4) Heatmaps True / Pred / |Error| ----------
def heatmaps_true_pred_error(t_list=None, fname_prefix="heatmap_TPE"):
    if t_list is None:
        t_list = [float(0.25*(t_min+t_max)), float(0.5*(t_min+t_max))]
    A = age_grid; X = xi_grid
    for t_phys in t_list:
        Up = predict_U_phys_on_grid(t_phys, A=A, X=X)         # (Na,Nx,5)
        it = nearest_time_index(t_phys)
        Ut = U_snaps[it,:,:,:]                                # (Na,Nx,5)
        Err = np.abs(Up - Ut)
        fig, axes = plt.subplots(K, 3, figsize=(12, 2.2*K))
        for j in range(K):
            im0 = axes[j,0].imshow(Ut[:,:,j].T, origin='lower', aspect='auto',
                                   extent=[A.min(),A.max(), X.min(),X.max()], cmap='viridis')
            axes[j,0].set_title(f"{comp_names[j]} true (t={ (t_phys-t_min)/(t_max-t_min+1e-12):.2f})");
            axes[j,0].set_xlabel("a"); axes[j,0].set_ylabel("ξ");
            plt.colorbar(im0, ax=axes[j,0], fraction=0.046, pad=0.04)

            im1 = axes[j,1].imshow(Up[:,:,j].T, origin='lower', aspect='auto',
                                   extent=[A.min(),A.max(), X.min(),X.max()], cmap='viridis')
            axes[j,1].set_title(f"{comp_names[j]} predicted"); axes[j,1].set_xlabel("a"); axes[j,1].set_ylabel("ξ")
            plt.colorbar(im1, ax=axes[j,1], fraction=0.046, pad=0.04)

            im2 = axes[j,2].imshow(Err[:,:,j].T, origin='lower', aspect='auto',
                                   extent=[A.min(),A.max(), X.min(),X.max()], cmap='magma')
            axes[j,2].set_title(f"|Error| {comp_names[j]}"); axes[j,2].set_xlabel("a"); axes[j,2].set_ylabel("ξ")
            plt.colorbar(im2, ax=axes[j,2], fraction=0.046, pad=0.04)
        plt.tight_layout()
        fname = f"{fname_prefix}_t{ (t_phys-t_min)/(t_max-t_min+1e-12):.2f}.png"
        plt.savefig(fname, dpi=160, bbox_inches='tight'); plt.show()
        print("Saved:", fname)

# ---------- 5) PDE residual heatmap ----------
def residual_heatmap(t_list=None, eps_x=1e-3, fname_prefix="heatmap_residual"):
    if t_list is None:
        t_list = [float(0.25*(t_min+t_max)), float(0.5*(t_min+t_max))]
    A = age_grid; X = xi_grid
    # prepare normalized grids
    a_ = torch.tensor((A - a_min)/((a_max-a_min)+1e-12), dtype=torch.float32, device=device).reshape(-1,1)
    x_ = torch.tensor((X - x_min)/((x_max-x_min)+1e-12), dtype=torch.float32, device=device).reshape(-1,1)
    aa, xx = torch.meshgrid(a_.flatten(), x_.flatten(), indexing='ij')

    for t_phys in t_list:
        t_ = torch.full_like(aa, (t_phys - t_min)/((t_max - t_min)+1e-12))
        with torch.enable_grad():
            # split-grads + Uxx (centered FD)
            def _pde_terms_split_eval(tn, an, xn):
                t_rng = (t_max - t_min) + 1e-12
                a_rng = (a_max - a_min) + 1e-12
                x_rng = (x_max - x_min) + 1e-12

                # dU/dt
                tt = tn.clone().requires_grad_(True)
                Ustd = model(torch.cat([tt, an.detach(), xn.detach()],1))
                Ut = []
                for k in range(K):
                    gk = torch.autograd.grad(Ustd[:,k].sum(), tt, create_graph=False, retain_graph=(k<K-1))[0]
                    Ut.append(gk)
                Ut = torch.cat(Ut,1) * y_std / t_rng

                # dU/da
                aa2 = an.clone().requires_grad_(True)
                Ustd = model(torch.cat([tn.detach(), aa2, xn.detach()],1))
                Ua = []
                for k in range(K):
                    gk = torch.autograd.grad(Ustd[:,k].sum(), aa2, create_graph=False, retain_graph=(k<K-1))[0]
                    Ua.append(gk)
                Ua = torch.cat(Ua,1) * y_std / a_rng

                # dU/dx
                xx2 = xn.clone().requires_grad_(True)
                Ustd = model(torch.cat([tn.detach(), an.detach(), xx2],1))
                Ux = []
                for k in range(K):
                    gk = torch.autograd.grad(Ustd[:,k].sum(), xx2, create_graph=False, retain_graph=(k<K-1))[0]
                    Ux.append(gk)
                Ux = torch.cat(Ux,1) * y_std / ((x_max - x_min)+1e-12)

                # U (phys) & Uxx (FD)
                U_phys = (Ustd * y_std + y_mean)
                x_plus  = torch.clamp(xn.detach() + eps_x, 0.0, 1.0)
                x_minus = torch.clamp(xn.detach() - eps_x, 0.0, 1.0)
                with torch.no_grad():
                    Ustd_p = model(torch.cat([tn.detach(), an.detach(), x_plus], 1))
                    Ustd_m = model(torch.cat([tn.detach(), an.detach(), x_minus],1))
                Uxx_norm = (Ustd_p - 2.0*Ustd.detach() + Ustd_m) / (eps_x**2)
                Uxx = (Uxx_norm * y_std) / (((x_max - x_min)+1e-12)**2)
                return Ut, Ua, Ux, Uxx, U_phys

            N = aa.numel()
            tn = t_.reshape(-1,1); an = aa.reshape(-1,1); xn = xx.reshape(-1,1)
            Ut,Ua,Ux,Uxx,U = _pde_terms_split_eval(tn,an,xn)
            AU = U @ A_fixed.T
            r  = Ut + Ua - d_exp*Uxx + v_exp*Ux - AU
            rL2 = torch.sqrt((r**2).sum(1)).detach().cpu().numpy().reshape(len(A), len(X))

        fig, ax = plt.subplots(1,1, figsize=(6,3))
        im = ax.imshow(rL2.T, origin='lower', aspect='auto',
                       extent=[A.min(),A.max(), X.min(),X.max()], cmap='magma')
        ax.set_title(f"‖PDE residual‖ (t={ (t_phys-t_min)/(t_max-t_min+1e-12):.2f})")
        ax.set_xlabel("a"); ax.set_ylabel("ξ")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        fname = f"{fname_prefix}_t{ (t_phys-t_min)/(t_max-t_min+1e-12):.2f}.png"
        plt.savefig(fname, dpi=160, bbox_inches='tight'); plt.show()
        print("Saved:", fname)

# ========== Run a few useful figures ==========
# (You can adjust the list of times)
times_for_lines = [float(0.00*(t_max-t_min)+t_min),
                   float(0.25*(t_max-t_min)+t_min),
                   float(0.50*(t_max-t_min)+t_min),
                   float(0.75*(t_max-t_min)+t_min),
                   float(1.00*(t_max-t_min)+t_min)]
xi_mid = float(0.5*(x_min+x_max))

plot_lines_vs_age_times(times_for_lines, xi_mid, fname="lines_PINN_S-D-T-A-P.png")
plot_lines_vs_age_times_compare([times_for_lines[1], times_for_lines[2], times_for_lines[3]],
                                xi_mid, fname="lines_compare_true_vs_pred_S-D-T-A-P.png")
heatmaps_true_pred_error([times_for_lines[1], times_for_lines[2]], fname_prefix="heatmap_T-P-E_S-D-T-A-P")
residual_heatmap([times_for_lines[2]], eps_x=1e-3, fname_prefix="heatmap_residual_S-D-T-A-P")

# ---------- Heatmaps True / Predicted / Error, 9x9 panel ----------
import matplotlib.pyplot as plt

COMP_NAMES = ["S","D","T","A","P"]

def _interp_snap_at_time(t_phys):
    """
    Returns a 'U_true' field (Na, Nx, 5) at physical time t_phys,
    by linearly interpolating between neighboring snapshots.
    """
    if t_phys <= times[0]:
        return U_snaps[0]
    if t_phys >= times[-1]:
        return U_snaps[-1]
    i = int(np.searchsorted(times, t_phys) - 1)
    i = max(0, min(len(times)-2, i))
    t0, t1 = float(times[i]), float(times[i+1])
    w = (t_phys - t0) / (t1 - t0 + 1e-12)
    return (1.0 - w) * U_snaps[i] + w * U_snaps[i+1]   # (Na, Nx, 5)

def _predict_grid_at_time(t_phys):
    """
    Model prediction on the (a, xi) grid at time t_phys.
    Returns (Na, Nx, 5) in physical units (de-standardized with y_std / y_mean).
    """
    Na = len(age_grid); Nx = len(xi_grid)
    # coordinate normalization
    t_n = (t_phys - float(times.min())) / (float(times.max() - times.min()) + 1e-12)
    a_n = (age_grid - float(age_grid.min())) / (float(age_grid.max() - age_grid.min()) + 1e-12)
    x_n = (xi_grid  - float(xi_grid.min()))  / (float(xi_grid.max()  - xi_grid.min())  + 1e-12)

    a_t = torch.tensor(a_n, dtype=torch.float32, device=device).view(-1,1)
    x_t = torch.tensor(x_n, dtype=torch.float32, device=device).view(1,-1)
    aa, xx = torch.meshgrid(a_t.flatten(), x_t.flatten(), indexing="ij")
    tt = torch.full_like(aa, float(t_n))

    with torch.no_grad():
        Ustd = model(torch.cat([tt.reshape(-1,1), aa.reshape(-1,1), xx.reshape(-1,1)], dim=1))
        Uphy = (Ustd * y_std + y_mean).reshape(Na, Nx, 5)
    return Uphy.detach().cpu().numpy()

def heatmaps_true_pred_error(times_years=[0.25, 1.0], days_per_year=365.0,
                             figsize=(9,9), save_prefix="heatmap_TPE"):
    """
    For each requested time (in years), plots a 5x3 panel (rows=compartments, columns=True/Predicted/Error)
    - Titles: 'S True', 'S Predicted', 'S Error', etc.
    - Panel size: 9x9 inches (default)
    - Color scales: True & Predicted share the same vmin/vmax for each compartment; Error uses its own scale [0, max].
    """
    A = age_grid
    X = xi_grid

    for t_yrs in times_years:
        t_phys = float(t_yrs * days_per_year)  # conversion to days if 'times' is in days
        # Truth and prediction
        U_true = _interp_snap_at_time(t_phys)             # (Na, Nx, 5)
        U_pred = _predict_grid_at_time(t_phys)            # (Na, Nx, 5)
        U_err  = np.abs(U_pred - U_true)                  # (Na, Nx, 5)

        # 9x9 figure with 5 rows x 3 columns
        fig, axes = plt.subplots(nrows=5, ncols=3, figsize=figsize,
                                 constrained_layout=True, sharex=False, sharey=False)

        for k, name in enumerate(COMP_NAMES):
            # shared bounds for True/Predicted for compartment k
            vmin = float(min(U_true[:,:,k].min(), U_pred[:,:,k].min()))
            vmax = float(max(U_true[:,:,k].max(), U_pred[:,:,k].max()))
            # True
            im0 = axes[k,0].imshow(U_true[:,:,k].T, origin='lower', aspect='auto',
                                   extent=[A.min(), A.max(), X.min(), X.max()],
                                   vmin=vmin, vmax=vmax, cmap='viridis')
            axes[k,0].set_title(f"{name} True")
            fig.colorbar(im0, ax=axes[k,0], fraction=0.046, pad=0.04)

            # Predicted
            im1 = axes[k,1].imshow(U_pred[:,:,k].T, origin='lower', aspect='auto',
                                   extent=[A.min(), A.max(), X.min(), X.max()],
                                   vmin=vmin, vmax=vmax, cmap='viridis')
            axes[k,1].set_title(f"{name} Predicted")
            fig.colorbar(im1, ax=axes[k,1], fraction=0.046, pad=0.04)

            # Error (own scale)
            im2 = axes[k,2].imshow(U_err[:,:,k].T, origin='lower', aspect='auto',
                                   extent=[A.min(), A.max(), X.min(), X.max()],
                                   vmin=0.0, vmax=float(U_err[:,:,k].max()+1e-12), cmap='magma')
            axes[k,2].set_title(f"{name} Error")
            fig.colorbar(im2, ax=axes[k,2], fraction=0.046, pad=0.04)

            # Axis labels
            for j in range(3):
                axes[k,j].set_xlabel("Age (days)")
            axes[k,0].set_ylabel("Exposure ξ")

        fig.suptitle(f"True / Predicted / Error at t = {t_yrs:.2f} years", y=1.02, fontsize=12)
        out = f"{save_prefix}_t{t_yrs:.2f}y.png"
        plt.savefig(out, dpi=160, bbox_inches="tight")
        plt.show()
        print("Saved:", out)

In [ ]:
# Panel at t = 0.25 year and t = 1.0 year, 9x9 format
heatmaps_true_pred_error(times_years=[0.25, 1.0], figsize=(9,8), save_prefix="heatmap_TPE")